In [12]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [13]:
from dotenv import load_dotenv
import os 
from tavily import TavilyClient
from langchain.chat_models import init_chat_model
from sqlalchemy import create_engine, text
from langchain.tools import tool
load_dotenv()

True

In [14]:
GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
DATABASE_URL = os.getenv("DATABASE_URL")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [15]:
# Fast + Free
cheap_model = init_chat_model(
    "google_genai:gemini-2.5-flash"
)

# Research + General Tasks
middle_model = init_chat_model(
    "google_genai:gemini-2.5-flash"
)

# Advanced Reasoning
advance_model = init_chat_model(
    "groq:llama-3.3-70b-versatile"
).with_fallbacks([middle_model, cheap_model])

In [16]:
from sqlalchemy import create_engine, text
from langchain_core.tools import tool

engine = create_engine(DATABASE_URL)

In [17]:
# user this information for testing porpose 
user_id_testing = "f5f7dea2-d2f9-431c-8529-aea5cd0fa49a"
user_mail = "nofackai@gmail.com"

In [18]:
import os
from dotenv import load_dotenv
import requests
from pinecone import Pinecone

# 1. Load API Keys from environment variables
load_dotenv(override=True)
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_index_name = os.getenv("PINECONE_INDEX", "outreachx")

# Provide the user_id if you want to search within a specific user's uploaded assets
# Your system saves data under the namespace format: "user_{user_id}"
user_id = user_id_testing

# 2. Initialize Pinecone Client
pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index(pinecone_index_name)

In [19]:
import json
from sqlalchemy import text
import requests

def get_user_profile(user_id: str) -> dict:
    """Fetches the user's basic profile from the database."""
    query = text("""
        SELECT full_name, email, role
        FROM users
        WHERE id = CAST(:user_id AS UUID);
    """)
    with engine.connect() as conn:
        result = conn.execute(query, {"user_id": user_id})
        row = result.fetchone()
        if row:
            data = dict(row._mapping)
            return {"name": data["full_name"], "role": data["role"], "email": data["email"]}
    return {"name": "User", "role": "Professional", "email": ""}

def get_conversation_history(user_id: str) -> list:
    """Fetches previous conversation history from the ai_memory table."""
    query = text("""
        SELECT role, content
        FROM ai_memory
        WHERE user_id = CAST(:user_id AS UUID)
        ORDER BY created_at DESC
        LIMIT 10;
    """)
    with engine.connect() as conn:
        result = conn.execute(query, {"user_id": user_id})
        rows = [dict(row._mapping) for row in result]
        return list(reversed(rows)) # Return chronological order

def retrieve_user_assets(user_id: str, query: str) -> list:
    """Searches Pinecone for user assets matching the query."""
    print(f"-> [RAG Tool] Retrieving assets for query: '{query}'")
    namespace = f"user_{user_id}"
    
    url = "https://api.cohere.com/v2/embed"
    payload = {
        "model": "embed-english-v3.0",
        "input_type": "search_query",
        "texts": [query],
        "output_dimension": 1024,
        "embedding_types": ["float"]
    }
    headers = {
        "Authorization": f"Bearer {COHERE_API_KEY}",
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        query_embedding = response.json().get("embeddings", {}).get("float")[0]
        
        search_results = index.query(
            vector=query_embedding,
            top_k=5,
            include_metadata=True,
            namespace=namespace
        )
        
        retrieved = [match['metadata'] for match in search_results.get('matches', [])]
        return retrieved
    except Exception as e:
        print(f"RAG Retrieval Error: {e}")
        return []

In [20]:
from typing import TypedDict, List, Dict, Any, Literal
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from pydantic import BaseModel, Field

# 1. State Definition
class GeneralAgentState(MessagesState):
    user_id: str
    basic_profile: Dict[str, Any]
    history: List[Dict[str, Any]]
    rag_query: str
    rag_results: List[Dict[str, Any]]
    routing_decision: str

# 2. Nodes
def context_loader(state: GeneralAgentState):
    """Loads the basic profile and memory."""
    print("-> [System] Loading basic profile and conversation memory...")
    profile = get_user_profile(state["user_id"])
    history = get_conversation_history(state["user_id"])
    return {"basic_profile": profile, "history": history}

# Pydantic Model for Routing Decision
class RouterDecision(BaseModel):
    decision: Literal["direct_reply", "needs_rag", "external_transfer"] = Field(
        description="Choose 'direct_reply' if the user is just saying hi or asking something already known. Choose 'needs_rag' if the user asks about their projects, skills, or specific past assets that require deep retrieval. Choose 'external_transfer' if they ask about real-world external knowledge."
    )
    optimized_rag_query: str = Field(
        description="If decision is 'needs_rag', provide a highly optimized search query to find the relevant user assets. Otherwise, leave blank.", default=""
    )

def decision_router(state: GeneralAgentState):
    """Uses a small LLM to decide the next step."""
    print("-> [Router] Analyzing intent...")
    messages = state["messages"]
    user_query = messages[-1].content
    profile = state.get("basic_profile", {})
    
    prompt = f"""
    You are the Routing Layer of the General Agent.
    User Query: '{user_query}'
    User Profile: Name: {profile.get('name')}, Role: {profile.get('role')}
    
    Determine if this query can be answered via basic profile/memory (direct_reply), or if it requires searching the user's specific uploaded files, projects, and campaign history (needs_rag).
    """
    
    structured_router = cheap_model.with_structured_output(RouterDecision)
    try:
        response = structured_router.invoke([HumanMessage(content=prompt)])
        decision = response.decision
        query = response.optimized_rag_query
    except Exception as e:
        decision = "direct_reply"
        query = ""
        
    print(f"-> [Router] Decision: {decision}")
    return {"routing_decision": decision, "rag_query": query}

def route_next_step(state: GeneralAgentState):
    """Conditional edge routing."""
    decision = state["routing_decision"]
    if decision == "needs_rag":
        return "rag_retriever"
    return "response_generator"

def rag_retriever(state: GeneralAgentState):
    """Retrieves assets if RAG is needed."""
    query = state["rag_query"]
    results = retrieve_user_assets(state["user_id"], query)
    return {"rag_results": results}

def response_generator(state: GeneralAgentState):
    """Generates the final response based on all context."""
    print("-> [Generator] Crafting final response...")
    
    user_query = state["messages"][-1].content
    profile = state["basic_profile"]
    history = state.get("history", [])
    rag_results = state.get("rag_results", [])
    decision = state["routing_decision"]
    
    if decision == "external_transfer":
        return {"messages": [AIMessage(content="I can help with your campaigns, outreach, and user assets. For external web research, please query the Research Agent!")]}
        
    system_prompt = f"""
    You are the General Agent of OutreachX Deva.
    Your responsibility is handling general conversations, memory-aware responses, and user-specific knowledge retrieval.
    
    CRITICAL RULES:
    1. NEVER hallucinate user data. If the answer is not in the context or profile, say you don't know.
    2. NEVER use external search tools.
    3. Maintain a helpful, conversational tone.
    
    USER PROFILE:
    Name: {profile.get('name')}
    Role: {profile.get('role')}
    
    DB CONVERSATION HISTORY:
    {history}
    
    RETRIEVED RAG KNOWLEDGE (if any):
    {rag_results}
    """
    
    try:
        response = advance_model.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_query)
        ])
        return {"messages": [response]}
    except Exception as e:
        print(f"Error in response generation: {e}")
        return {"messages": [AIMessage(content="Sorry, I encountered an error generating the response.")]}

# 3. Compile Graph
workflow = StateGraph(GeneralAgentState)

workflow.add_node("loader", context_loader)
workflow.add_node("router", decision_router)
workflow.add_node("rag_retriever", rag_retriever)
workflow.add_node("response_generator", response_generator)

workflow.add_edge(START, "loader")
workflow.add_edge("loader", "router")
workflow.add_conditional_edges("router", route_next_step)
workflow.add_edge("rag_retriever", "response_generator")
workflow.add_edge("response_generator", END)

general_agent = workflow.compile()
print("General Agent Workflow Compiled Successfully!")

General Agent Workflow Compiled Successfully!


In [21]:
# ==========================================
# 4. Interactive Testing Examples
# ==========================================

print("=== EXAMPLE 1: NORMAL CONVERSATION ===")
ex1_state = {
    "user_id": user_id_testing,
    "messages": [HumanMessage(content="Hi")]
}
result1 = general_agent.invoke(ex1_state)
print(f"\nAgent: {result1['messages'][-1].content}\n")
print("="*50)


print("\n=== EXAMPLE 2: USER ASKS ABOUT OWN PROJECT ===")
ex2_state = {
    "user_id": user_id_testing,
    "messages": [HumanMessage(content="In which project did I use LangGraph?")]
}
result2 = general_agent.invoke(ex2_state)
print(f"\nAgent: {result2['messages'][-1].content}\n")
print("="*50)

print("\n=== EXAMPLE 3: USER DISSATISFACTION ===")
ex3_state = {
    "user_id": user_id_testing,
    "messages": [HumanMessage(content="I am not satisfied with the previous campaign template we made.")]
}
result3 = general_agent.invoke(ex3_state)
print(f"\nAgent: {result3['messages'][-1].content}\n")
print("="*50)

=== EXAMPLE 1: NORMAL CONVERSATION ===
-> [System] Loading basic profile and conversation memory...
-> [Router] Analyzing intent...
-> [Router] Decision: direct_reply
-> [Generator] Crafting final response...

Agent: Hello Moksh Bhardwaj. It's nice to meet you again. I see we had a previous conversation, but you had asked me to clear the history, so we're starting fresh. How can I assist you today?


=== EXAMPLE 2: USER ASKS ABOUT OWN PROJECT ===
-> [System] Loading basic profile and conversation memory...
-> [Router] Analyzing intent...
-> [Router] Decision: needs_rag
-> [RAG Tool] Retrieving assets for query: 'projects using LangGraph'
-> [Generator] Crafting final response...

Agent: I don't know which project you used LangGraph in. That information is not in our conversation history or your profile. If you'd like to share more about the project, I'd be happy to help with any questions or topics you'd like to discuss.


=== EXAMPLE 3: USER DISSATISFACTION ===
-> [System] Loading bas